In [17]:
import os
import cv2
import numpy as np
from ultralytics import YOLO
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from collections import defaultdict

CHAR_MODEL = YOLO("models/OCR_MODEL.pt")
IMG_DIR = "../data/raw/Indonesian License Plate Recognition Dataset/images/test"
LBL_DIR = "../data/raw/Indonesian License Plate Recognition Dataset/labels/test"

IOU_TH = 0.5
CONF_TH = 0.25

def iou(box1, box2):
    xi1 = max(box1[0], box2[0])
    yi1 = max(box1[1], box2[1])
    xi2 = min(box1[2], box2[2])
    yi2 = min(box1[3], box2[3])
    inter = max(0, xi2-xi1)*max(0, yi2-yi1)
    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

y_true, y_pred = [], []

for img_name in os.listdir(IMG_DIR):
    img_path = os.path.join(IMG_DIR, img_name)
    label_path = os.path.join(LBL_DIR, img_name.replace(".jpg", ".txt"))

    img = cv2.imread(img_path)
    h, w, _ = img.shape

    gt_boxes, gt_cls = [], []
    with open(label_path) as f:
        for l in f:
            c, xc, yc, bw, bh = map(float, l.split())
            x1 = (xc - bw/2) * w
            y1 = (yc - bh/2) * h
            x2 = (xc + bw/2) * w
            y2 = (yc + bh/2) * h
            gt_boxes.append([x1,y1,x2,y2])
            gt_cls.append(int(c))

    preds = CHAR_MODEL.predict(img, conf=CONF_TH, verbose=False)[0]
    used = set()

    for p in preds.boxes:
        pb = p.xyxy[0].cpu().numpy()
        pc = int(p.cls[0])

        best_iou, best_idx = 0, -1
        for i, gb in enumerate(gt_boxes):
            if i in used: continue
            cur_iou = iou(pb, gb)
            if cur_iou > best_iou:
                best_iou, best_idx = cur_iou, i

        if best_iou >= IOU_TH:
            used.add(best_idx)
            y_true.append(gt_cls[best_idx])
            y_pred.append(pc)

# Metrics
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)

print("Character Accuracy:", acc)
print("Precision (macro):", prec)
print("Recall (macro):", rec)
print("F1-score:", f1)


Character Accuracy: 0.9699720670391061
Precision (macro): 0.957801173358448
Recall (macro): 0.9497501075965589
F1-score: 0.9509828127610598


In [28]:
metrics = CHAR_MODEL.val(
    data="data_char_recognition.yaml",
    split="test",
    conf=0.25)

print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.3.237 🚀 Python-3.10.19 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
val: Fast image access ✅ (ping: 1.7±0.2 ms, read: 10.3±1.9 MB/s, size: 25.9 KB)
val: Scanning /mnt/d/Uni/Semester 5/Computer Vision/Plate-Recognition-backend-notebooks/data/raw/Indonesian License Plate Recognition Dataset/labels/test... 197 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 197/197 852.5it/s 0.2s0.1s
val: New cache created: /mnt/d/Uni/Semester 5/Computer Vision/Plate-Recognition-backend-notebooks/data/raw/Indonesian License Plate Recognition Dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 5.7it/s 2.3s0.2s
                   all        197       1455      0.967      0.976      0.981      0.645
                     0         59         62      0.954          1      0.992      0.717
                     1        139        173      0.989      0.994      0.995      0.563
    

In [3]:
PLATE_MODEL = YOLO("models/LPR_MODEL.pt")
metrics = PLATE_MODEL.val(
    data="../data.yaml",     # same YAML used in training
    split="test",         # IMPORTANT: test split
    conf=0.25,           # low conf for proper PR curve
    iou=0.5,
    device=0
)


Ultralytics 8.3.237 🚀 Python-3.10.19 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients
val: Fast image access ✅ (ping: 1.6±0.1 ms, read: 130.1±21.2 MB/s, size: 1595.7 KB)
val: Scanning /mnt/d/Uni/Semester 5/Computer Vision/Plate-Recognition-backend-notebooks/data/processed/labels/test... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 823.5it/s 0.1s1s
val: New cache created: /mnt/d/Uni/Semester 5/Computer Vision/Plate-Recognition-backend-notebooks/data/processed/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.0it/s 3.6s0.3ss
                   all        100        197      0.974      0.958      0.981       0.73
Speed: 7.0ms preprocess, 6.9ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to /mnt/d/Uni/Semester 5/Computer Vision/Plate-Recognition-backend-notebooks/notebooks/runs/det

In [4]:
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)


mAP@0.5: 0.9809916232817999
mAP@0.5:0.95: 0.729981726461502
Precision: 0.9741831995973809
Recall: 0.9577333597209506


In [5]:
import os, cv2, numpy as np

IMG_DIR = "../data/raw/Indonesian License Plate Dataset/images/test"
LABEL_DIR = "../data/raw/Indonesian License Plate Dataset/labels/test"

def yolo_to_xyxy(b, w, h):
    xc, yc, bw, bh = b
    return np.array([
        (xc - bw/2) * w,
        (yc - bh/2) * h,
        (xc + bw/2) * w,
        (yc + bh/2) * h
    ])

def compute_iou(a, b):
    xi1, yi1 = max(a[0], b[0]), max(a[1], b[1])
    xi2, yi2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-6)


In [7]:
ious = []

for img_name in os.listdir(IMG_DIR):
    if not img_name.endswith((".jpg", ".png")):
        continue

    img_path = os.path.join(IMG_DIR, img_name)
    label_path = os.path.join(LABEL_DIR, img_name.replace(".jpg", ".txt"))

    if not os.path.exists(label_path):
        continue

    img = cv2.imread(img_path)
    h, w, _ = img.shape

    # Load GT boxes
    gt_boxes = []
    with open(label_path) as f:
        for line in f:
            _, xc, yc, bw, bh = map(float, line.split())
            gt_boxes.append(yolo_to_xyxy((xc,yc,bw,bh), w, h))

    # Predict
    preds = PLATE_MODEL.predict(img, conf=0.25, verbose=False)[0].boxes

    for gt in gt_boxes:
        best_iou = 0
        for p in preds:
            pred_box = p.xyxy[0].cpu().numpy()
            best_iou = max(best_iou, compute_iou(gt, pred_box))

        if best_iou > 0:
            ious.append(best_iou)

print("Mean IoU:", sum(ious) / len(ious))


Mean IoU: 0.8528357284098523


End to end evaluation

In [18]:
CLASS_NAMES = [
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J',
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T',
    'U', 'V', 'W', 'X', 'Y', 'Z'
]

In [19]:
def levenshtein(a, b):
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost  # substitution
            )
    return dp[n][m]


In [ ]:
import os
import time
import cv2
import numpy as np
from ultralytics import YOLO

# -----------------------------
# Models
# -----------------------------
CHAR_MODEL  = YOLO("models/best.pt")  # ganti sesuai path model karakter

# -----------------------------
# Paths
# -----------------------------
IMG_DIR = "../data/raw/Indonesian License Plate Dataset/images/test/"
PLATE_LABEL_DIR = "../data/raw/Indonesian License Plate Dataset/labels/test/"
CHAR_LABEL_DIR  = "../data/raw/Indonesian License Plate Recognition Dataset/labels/test"
# -----------------------------
# Config
# -----------------------------
CONF_TH = 0.25
IOU_TH  = 0.5

CLASS_NAMES = [
    '0','1','2','3','4','5','6','7','8','9',
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z'
]
ID2CHAR = {i: c for i, c in enumerate(CLASS_NAMES)}

# -----------------------------
# Utils
# -----------------------------
def yolo_to_xyxy(b, w, h):
    xc, yc, bw, bh = b
    return np.array([
        (xc - bw/2) * w,
        (yc - bh/2) * h,
        (xc + bw/2) * w,
        (yc + bh/2) * h
    ])

def iou(a, b):
    xi1, yi1 = max(a[0], b[0]), max(a[1], b[1])
    xi2, yi2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    area1 = (a[2]-a[0])*(a[3]-a[1])
    area2 = (b[2]-b[0])*(b[3]-b[1])
    return inter / (area1 + area2 - inter + 1e-6)

def load_gt_chars(char_label_path):
    chars = []
    with open(char_label_path) as f:
        for line in f:
            cls, xc, yc, bw, bh = map(float, line.split())
            chars.append((xc, ID2CHAR[int(cls)]))
    chars.sort(key=lambda x: x[0])  # left → right
    return "".join(c[1] for c in chars)

# -----------------------------
# Evaluation
# -----------------------------
correct = 0
total   = 0
times   = []
total_char_errors = 0
total_gt_chars = 0


for img_name in os.listdir(IMG_DIR):
    if not img_name.endswith((".jpg", ".png")):
        continue

    img_path = os.path.join(IMG_DIR, img_name)
    plate_label_path = os.path.join(
        PLATE_LABEL_DIR, img_name.replace(".jpg", ".txt")
    )

    if not os.path.exists(plate_label_path):
        continue

    img = cv2.imread(img_path)
    h, w, _ = img.shape

    # Load GT plates
    gt_plates = []
    with open(plate_label_path) as f:
        for line in f:
            _, xc, yc, bw, bh = map(float, line.split())
            gt_plates.append(yolo_to_xyxy((xc, yc, bw, bh), w, h))

    # Inference
    start = time.time()
    pred_plates = PLATE_MODEL.predict(img, conf=CONF_TH, verbose=False)[0].boxes

    for idx, gt_plate in enumerate(gt_plates, start=1):
        best_iou, best_pred = 0, None

        for pb in pred_plates:
            pred_box = pb.xyxy[0].cpu().numpy()
            cur_iou = iou(gt_plate, pred_box)
            if cur_iou > best_iou:
                best_iou, best_pred = cur_iou, pred_box

        total += 1
        if best_iou < IOU_TH:
            continue

        # Crop plate
        x1,y1,x2,y2 = map(int, best_pred)
        roi = img[y1:y2, x1:x2]

        # Predict characters
        char_preds = CHAR_MODEL.predict(roi, conf=CONF_TH, verbose=False)[0].boxes
        pred_chars = [(b.xyxy[0][0].item(), ID2CHAR[int(b.cls[0])]) for b in char_preds]
        pred_chars.sort(key=lambda x: x[0])
        pred_text = "".join(c[1] for c in pred_chars)

        # Load GT chars
        base = img_name.replace(".jpg", "")
        char_label_path = os.path.join(CHAR_LABEL_DIR, f"{base}_{idx}.txt")

        if not os.path.exists(char_label_path):
            continue

        gt_text = load_gt_chars(char_label_path)
        edit_dist = levenshtein(pred_text, gt_text)

        total_char_errors += edit_dist
        total_gt_chars += len(gt_text)

        print(f"[{img_name} | Plate {idx}]")
        print("Pred:", pred_text)
        print("GT  :", gt_text)
        print("-"*30)

        if pred_text == gt_text:
            correct += 1

    times.append(time.time() - start)

# -----------------------------
# Results
# -----------------------------
acc = correct / total if total > 0 else 0
avg_time = sum(times) / len(times)
fps = 1 / avg_time
cer = total_char_errors / (total_gt_chars + 1e-6)

print("End-to-End Character Error Rate (CER):", cer)
print("✅ End-to-End Accuracy:", acc)
print("⏱ Avg Inference Time (ms):", avg_time * 1000)
print("🚀 FPS:", fps)

[test001.jpg | Plate 1]
Pred: B9140BCD
GT  : B9140BCD
------------------------------
[test001.jpg | Plate 2]
Pred: B2407UZO
GT  : B2407UZO
------------------------------
[test001.jpg | Plate 3]
Pred: B2842PKM
GT  : B2842PKM
------------------------------
[test002.jpg | Plate 1]
Pred: BG1352AE
GT  : BG1352AE
------------------------------
[test003.jpg | Plate 1]
Pred: B2634UZF
GT  : B2634UZF
------------------------------
[test003.jpg | Plate 2]
Pred: B1995JVK
GT  : B1995JVK
------------------------------
[test004.jpg | Plate 1]
Pred: B39062VENH
GT  : B9062VEH
------------------------------
[test005.jpg | Plate 1]
Pred: DD3798KM
GT  : DD8798KM
------------------------------
[test006.jpg | Plate 1]
Pred: T1329KC
GT  : T1329KC
------------------------------
[test007.jpg | Plate 1]
Pred: AD8865EE
GT  : AD8865EE
------------------------------
[test008.jpg | Plate 1]
Pred: DK1157AAB
GT  : DK1157AAB
------------------------------
[test008.jpg | Plate 2]
Pred: AA1997FE
GT  : AA1997FE
---------